In [ ]:
import numpy as np
import rasterio as rio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path

In [3]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FEATURE_DIR = RESULTS / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

In [6]:
accessibility_1km_ = rio.open(DATA / 'accessibility_to_cities_2015_v1.0.tif')

In [7]:
accessibility_1km = accessibility_1km_.read(1)

In [8]:
accessibility_1km = np.where(accessibility_1km < 0, np.nan, accessibility_1km).astype(np.float32)

In [9]:
accessibility_1km = np.empty(country_raster_1km_.shape, dtype=np.float32)
reproject(
    source=accessibility_1km,
    destination=accessibility_1km,
    src_transform=accessibility_1km_.transform,
    src_crs=accessibility_1km_.crs,
    dst_transform=country_raster_1km_.transform,
    dst_crs=country_raster_1km_.crs,
    resampling=Resampling.nearest,
)

(array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], shape=(16922, 33844), dtype=float32),
 Affine(1000.0, 0.0, -16922000.0,
        0.0, -1000.0, 8461000.0))

In [10]:
with rio.open(
    FEATURE_DIR / 'accessibility-1km.tif',
    "w",
    driver="GTiff",
    height=accessibility_1km.shape[0],
    width=accessibility_1km.shape[1],
    count=1,
    dtype=accessibility_1km.dtype,
    crs=country_raster_1km_.crs,
    transform=country_raster_1km_.transform,
    compress='lzw'
) as dst:
    dst.write(accessibility_1km, 1)